# QDAC - NI DAQ, QCS M5200 Parallel Read

In this code, we connect to Keysight M5200 with Pyro4 RPC, and measure the signal with NI DAQ simultaneously.

## Import Libraries

In [1]:
import time
import json
import pyvisa
import numpy as np
import matplotlib.pyplot as plt
from time import sleep

from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)
from qcodes.instrument.channel import ChannelList

from qstl_instruments.qstl_qdac2 import QSTL_QDac2
from qstl_instruments.qstl_nidaq import QSTL_NIDaq
from qstl_instruments.sg386 import SG386
from qstl_instruments.open_proxy import make_proxy

from pyvisa.constants import StopBits, Parity

rm = pyvisa.ResourceManager()
rm.list_resources()

('USB0::0x0957::0x1780::MY60101437::INSTR',
 'ASRL1::INSTR',
 'ASRL3::INSTR',
 'ASRL5::INSTR',
 'ASRL6::INSTR',
 'ASRL20::INSTR',
 'ASRL21::INSTR',
 'ASRL22::INSTR',
 'ASRL23::INSTR',
 'ASRL24::INSTR',
 'ASRL25::INSTR',
 'ASRL26::INSTR',
 'ASRL27::INSTR',
 'GPIB0::27::INSTR')

## Instantiation of Instruments

In [2]:
contacts = {
    "C1" : 1,
    "C2" : 2
}

ai_chans = {
    "I1" : "Dev2/ai0",
    "I2" : "Dev2/ai1",
    "I3" : "Dev2/ai2"
}

# Set up database
initialise_or_create_database_at("C:/Users/Measurement6/Nextcloud2/Lab/Data/QSim/2025/20251009_QDAC_NIQAC_synctest/QDAC_IQ_Mod_Bandwidth_Test.db")

qdac2 = QSTL_QDac2(
    name = "qdac2",
    address = "ASRL5::INSTR",
    ramp_rate = 1,
    i_threshold = 2e-9,
    v_limit = 0.5,
    contacts = contacts
)
station = Station(qdac2)

daq = QSTL_NIDaq(
    max_sampling_rate = int(1e6),
    gain = 1e8
)

srs_rf = SG386(
    name = "SRS_SG386",
    address = "GPIB0::27::INSTR"
)

qcs_dig_0 = make_proxy(
    ns_host = "192.168.19.2",
    ns_port = 8888,
    proxy_name = "digitizer_0"
)

qcs_dig_1 = make_proxy(
    ns_host = "192.168.19.2",
    ns_port = 8888,
    proxy_name = "digitizer_1"
)

qdac2.ramp_all_channels_to_zero()
qdac2.get_initial_voltages()

Connected to: QDevil QDAC-II (serial:368, firmware:13-1.57) in 0.09s
Connected to: Stanford Research Systems SG386 (serial:s/n004342, firmware:ver1.50.26) in 0.02s
Pyro.NameServer PYRO:Pyro.NameServer@192.168.19.2:8888
digitizer_0 PYRO:obj_b17a9da8fd29415f80d857407648ccdf@192.168.19.2:51401
digitizer_1 PYRO:obj_0c49e13947a2471d801461531a557db0@192.168.19.2:51401
Pyro.NameServer PYRO:Pyro.NameServer@192.168.19.2:8888
digitizer_0 PYRO:obj_b17a9da8fd29415f80d857407648ccdf@192.168.19.2:51401
digitizer_1 PYRO:obj_0c49e13947a2471d801461531a557db0@192.168.19.2:51401


{'C1': 0.0, 'C2': 0.0}

## Instruments Setup

In [13]:
## setup qdac 2 for the 2D sweep
qdac2.free_all_triggers()
qdac2.ext3.delay_s(0)
qdac2.v_limit = 1.2

device1 = "I1"
device2 = "I2"

slow_chans = ["C1"]
fast_chans = ["C2"]

slow_start = 0.0
slow_end = 0.2
slow_steps = 100

fast_start = 0.0
fast_end = 0.2
fast_steps = 100
fast_step_time_s = 2e-6

qcs_sampling_rate = 4.8e9
qcs_dig_samples = ((int(qcs_sampling_rate * fast_step_time_s * fast_steps) // 16) << 4) - 16

print(f"QCS Digitizer Memory Size : {qcs_dig_samples * 2 / 1e9} GB")

if qcs_dig_samples * 2 > 1e9:
    raise ValueError("# of QCS Samples exceeds maximum memory size...")

qdac2.channels[0:23].dc_slew_rate_V_per_s("inf")

for i in [slow_start, slow_end, fast_start, fast_end]:
    qdac2.validate_voltages([i])

slow_vs = np.linspace(slow_start, slow_end, slow_steps, endpoint=True)
fast_vs = np.linspace(fast_start, fast_end, fast_steps, endpoint=True)

exp = load_or_create_experiment("2D sweep", "QDAC+NiDAQ_IQ_Samp_Bandwidth_Test")
meas = Measurement(exp=exp, station=station)

Vslow = Parameter(name="Vslow", label=str(slow_chans), unit="V")
Vfast = Parameter(name="Vfast", label=str(fast_chans), unit="V")

I = Parameter(name= "I", label="I", unit="V")
Q = Parameter(name= "Q", label="Q", unit="V")

I_qcs = Parameter(name= "I_qcs", label="I_qcs", unit="V")
Q_qcs = Parameter(name= "Q_qcs", label="Q_qcs", unit="V")

A = Parameter(name= "A", label="A", unit="V")
Phi = Parameter(name= "theta", label="theta", unit="deg")

A_qcs = Parameter(name= "A_qcs", label="A_qcs", unit="V")
Phi_qcs = Parameter(name= "theta_qcs", label="theta_qcs", unit="deg")

meas.register_parameter(Vfast)
meas.register_parameter(Vslow)
meas.register_parameter(I, setpoints = (Vfast, Vslow))
meas.register_parameter(Q, setpoints = (Vfast, Vslow))
meas.register_parameter(I_qcs, setpoints = (Vfast, Vslow))
meas.register_parameter(Q_qcs, setpoints = (Vfast, Vslow))
meas.register_parameter(A, setpoints = (Vfast, Vslow))
meas.register_parameter(Phi, setpoints = (Vfast, Vslow))
meas.register_parameter(A_qcs, setpoints = (Vfast, Vslow))
meas.register_parameter(Phi_qcs, setpoints = (Vfast, Vslow))

QCS Digitizer Memory Size : 0.001919968 GB


## Measurement (Multi Channel)

In [14]:
qdac2.channels[0:23].dc_slew_rate_V_per_s("inf")
qdac2.free_all_triggers()

sweep_time = fast_steps * fast_step_time_s
samples_per_fast_scan_per_channel = fast_steps * int(fast_step_time_s*daq.max_sampling_rate/2)

arrangement = qdac2.arrange(
    contacts= {x: contacts[x] for x in fast_chans},
    output_triggers={
        "NIDAQ" : 5,
        "QCS_DAQ" : 1,
    }
)

sweep = arrangement.virtual_detune(
    contacts = tuple(fast_chans),
    start_V = (fast_start,) * len(fast_chans),
    end_V = (fast_end,) * len(fast_chans),
    steps = fast_steps,
    step_trigger = "NIDAQ",
    step_time_s = fast_step_time_s,
    repetitions = 1
)

# Route NIDAQ trigger to QCS trigger
t = arrangement.get_trigger_by_name("NIDAQ")
qdac2.ext1.width_s(2e-6)
qdac2.ext1.polarity('norm')
qdac2.ext1.source_from_trigger(t)

InitialConditions = qdac2.get_initial_voltages()

start_time = time.time()
with meas.run() as datasaver:
    loop_counter = 0
    datasaver.dataset.add_metadata(tag="Contacts", metadata=json.dumps(contacts))
    datasaver.dataset.add_metadata(tag="IC", metadata=json.dumps(InitialConditions))
    datasaver.dataset.add_metadata(
        tag = "Sweep_params",
        metadata = json.dumps(
            {
                "slow_chans" : slow_chans,
                "slow_start" : slow_start,
                "slow_end" : slow_end,
                "fast_chans" : fast_chans,
                "fast_start" : fast_start,
                "fast_end" : fast_end,
                "fast_step_time_s" : fast_step_time_s
            }
        )
    )
    for slow_v in slow_vs:
        # Set QDAC Voltages
        qcs_dig_0.set_daq_config(qcs_dig_samples)
        qcs_dig_1.set_daq_config(qcs_dig_samples)
        qcs_dig_0.daq_flush()
        qcs_dig_1.daq_flush()
        qcs_dig_0.daq_start()
        qcs_dig_1.daq_start()

        qdac2.ramp_channels(slow_chans, [slow_v])
        # Read NI Daq traces
        result = daq.read_triggered_multi_channels(
            sweep,
            [ai_chans[device1], ai_chans[device2]],
            samples_per_fast_scan_per_channel,
            -1,
            +1,
            sweep_time+1
        )
        # Wait until QCS Sampling ends
        while qcs_dig_0.daq_counter != qcs_dig_samples:
            print(f"Current QCS Digitizer Sample Number : {qcs_dig_0.daq_counter}...")
        while qcs_dig_1.daq_counter != qcs_dig_samples:
            print(f"Current QCS Digitizer Sample Number : {qcs_dig_1.daq_counter}...")

        # Read QCS Digitizer Traces
        buffer = np.array([], dtype = np.int16)
        buffer = qcs_dig_0.fetch_waveform_int16(buffer, qcs_dig_samples)
        qcs_result_1 = daq.reshape_array(buffer, fast_steps)

        buffer = np.array([], dtype = np.int16)
        buffer = qcs_dig_1.fetch_waveform_int16(buffer, qcs_dig_samples)
        qcs_result_0 = daq.reshape_array(buffer, fast_steps)

        # Read NI DAQ Samples
        result_0 = daq.reshape_array(result[0,:], fast_steps)
        result_1 = daq.reshape_array(result[1,:], fast_steps)

        amp = np.abs(result_0+1j*result_1)
        phase = np.angle(result_0+1j*result_1, deg=True)
        amp_qcs = np.abs(qcs_result_0+1j*qcs_result_1)
        phase_qcs = np.angle(qcs_result_0+1j*qcs_result_1, deg=True)

        datasaver.add_result(
            (Vslow, [slow_v]*fast_steps),
            (Vfast, fast_vs),
            (I, result_0),
            (Q, result_1),
            (I_qcs, qcs_result_0),
            (Q_qcs, qcs_result_1),
            (A, amp),
            (Phi, phase),
            (A_qcs, amp_qcs),
            (Phi_qcs, phase_qcs)
        )  
            
        loop_counter = loop_counter+1
        print(f'Time elapsed: {np.round(time.time()-start_time, 2)} sec. Loop finished: {loop_counter}/{slow_steps}.')

end_time = time.time()
print(f'Time elapsed: {np.round(end_time-start_time, 2)} sec.')

qdac2.channels[0:23].dc_slew_rate_V_per_s(1)

qdac2.ramp_all_channels_to_zero()

Starting experimental run with id: 114. 
Time elapsed: 3.04 sec. Loop finished: 1/100.
Time elapsed: 4.06 sec. Loop finished: 2/100.
Time elapsed: 4.22 sec. Loop finished: 3/100.
Time elapsed: 4.38 sec. Loop finished: 4/100.
Time elapsed: 4.79 sec. Loop finished: 5/100.
Time elapsed: 4.95 sec. Loop finished: 6/100.
Time elapsed: 5.14 sec. Loop finished: 7/100.
Time elapsed: 5.56 sec. Loop finished: 8/100.
Time elapsed: 5.72 sec. Loop finished: 9/100.
Time elapsed: 6.12 sec. Loop finished: 10/100.
Time elapsed: 6.28 sec. Loop finished: 11/100.
Time elapsed: 6.44 sec. Loop finished: 12/100.
Time elapsed: 6.86 sec. Loop finished: 13/100.
Time elapsed: 7.01 sec. Loop finished: 14/100.
Time elapsed: 7.16 sec. Loop finished: 15/100.
Time elapsed: 7.32 sec. Loop finished: 16/100.
Time elapsed: 7.48 sec. Loop finished: 17/100.
Time elapsed: 7.63 sec. Loop finished: 18/100.
Time elapsed: 8.06 sec. Loop finished: 19/100.
Time elapsed: 8.21 sec. Loop finished: 20/100.
Time elapsed: 8.37 sec. Loop

## SRS RF Source Setup

In [3]:
srs_rf.enable_RF("OFF")
# srs_rf.frequency(470e6)

In [4]:
qdac2.reset()